In [ ]:
# First make sure Drive is mounted
from google.colab import drive
drive.mount('/content/drive')

# Search for the file
!find /content/drive -name "*Coff*" -o -name "*coffee*" -o -name "*Production*" 2>/dev/null | head -20

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Copy of Coffee_yeild_pred.ipynb
/content/drive/MyDrive/coffee proposal @ OBU.docx
/content/drive/MyDrive/my file/coffee proposal @ OBU.docx
/content/drive/MyDrive/Coffee_yeild_pred.csv
/content/drive/MyDrive/Coffee_yeild_Predc (1).csv
/content/drive/MyDrive/Coffee_yeild_Predc.csv
/content/drive/MyDrive/Coffee222.csv
/content/drive/MyDrive/Coffee111.csv
/content/drive/MyDrive/Coffe_Production.csv
/content/drive/MyDrive/Coff_Poduction.csv


In [52]:
import os, json, hashlib, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import linregress, gaussian_kde
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns

OUT = Path("publication_outputs_final")
(OUT / "tables").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial','Helvetica','DejaVu Sans'],
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 12,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'savefig.dpi': 600, 'savefig.bbox': 'tight', 'savefig.facecolor': 'white'
})
sns.set_style("whitegrid")

print("="*80)
print("FINAL REVISION — Year-blocked CV + group-aware bootstrap + baselines")
print("="*80)

FINAL REVISION — Year-blocked CV + group-aware bootstrap + baselines


In [53]:
# ==============================================================================
# PART 1 — DATA LOADING + DISTRICT MAPPING  [REV#2 district IDs retained]
# ==============================================================================
DATA_PATH = '/content/drive/MyDrive/Coff_Poduction.csv'
data = pd.read_csv(DATA_PATH, encoding='latin1')
print(f"Shape: {data.shape}")

# District column detection
DISTRICT_COL = None
for cand in ['District_Name','District','district','Woreda','woreda','WOREDA']:
    if cand in data.columns:
        DISTRICT_COL = cand; break
if DISTRICT_COL != 'District':
    data = data.rename(columns={DISTRICT_COL: 'District'})

# Standardized names  [minor: spellings]
DISTRICT_NAME_MAP = {
    'Boke':'Boke', 'D/Labu':'Dara Labu', 'Habro':'Habro',
    'Mesela':'Shanan_dhugoo', 'Tulo':'Tulo'
}
data['District'] = data['District'].map(DISTRICT_NAME_MAP).fillna(data['District'])

# Rename variables (same as original)
data = data.rename(columns={
    'Yeild(tone/hect)':'Production_tons',
    'Area(hec)':'Cultivated_Area_ha',
    'Tot. N (%)':'Total_N_percent',
    'Avail. P (ppm)':'Available_P_ppm',
    'Exch. K (ppm)':'Exchangeable_K_ppm',
    'Org. C (%)':'Organic_C_percent',
    'ANN_Temperature':'Annual_Temperature_C',
    'Precipitation(mm/day)':'Precipitation_mm_day',
    'Relative_humidity':'Relative_Humidity_percent',
    'ANN_ Wind Speed(m/s)':'Wind_Speed_m_s',
    'Soil wetness':'Soil_Wetness',
    'Soil Type':'Soil_Type',
    'Fertilizers':'Fertilizer_Used'
})
# Robustly rename solar column
for c in list(data.columns):
    if 'Solar' in c:
        data = data.rename(columns={c:'Solar_Radiation_MJ_m2_day'})

# Encode fertilizer binary
if data['Fertilizer_Used'].dtype == object:
    data['Fertilizer_Used'] = data['Fertilizer_Used'].map(
        {'yes':1,'no':0,'Yes':1,'No':0}).fillna(0)

data['Year'] = data['Year'].astype(int)
data = data.sort_values(['Year','District']).reset_index(drop=True)
print(f"Loaded N={len(data)}, districts={data['District'].nunique()}, "
      f"years {data['Year'].min()}–{data['Year'].max()}")

Shape: (500, 17)
Loaded N=500, districts=5, years 2000–2015


In [54]:
# ==============================================================================
# PART 2 — DATA DICTIONARY
# ==============================================================================
UNITS = {
    'Cultivated_Area_ha':'hectares','pH':'pH units','Total_N_percent':'%',
    'Available_P_ppm':'ppm','Exchangeable_K_ppm':'ppm','Organic_C_percent':'%',
    'Annual_Temperature_C':'°C','Precipitation_mm_day':'mm/day',
    'Relative_Humidity_percent':'%','Wind_Speed_m_s':'m/s',
    'Solar_Radiation_MJ_m2_day':'MJ/m²/day','Soil_Wetness':'dimensionless',
    'Production_tons':'metric tons'
}
dd_rows=[]
for col in data.columns:
    if col in ['District','Year']: continue
    s = data[col]
    if pd.api.types.is_numeric_dtype(s):
        dd_rows.append(dict(variable=col, type='numeric', units=UNITS.get(col,''),
            levels='', min=round(float(s.min()),3), max=round(float(s.max()),3),
            mean=round(float(s.mean()),3)))
    else:
        dd_rows.append(dict(variable=col, type='categorical', units='',
            levels='; '.join(sorted(map(str,s.unique()))),
            min='', max='', mean=''))
pd.DataFrame(dd_rows).to_csv(OUT/"tables/Table_S4_Data_Dictionary.csv", index=False)

In [56]:
# ==============================================================================
# PART 3 — DATA-QUALITY AUDIT
# ==============================================================================
rows=[]
for col in data.columns:
    if col in ['District','Year']: continue
    s = data[col]
    if pd.api.types.is_numeric_dtype(s):
        q1,q3 = s.quantile(.25), s.quantile(.75); iqr=q3-q1
        n_out = int(((s<q1-1.5*iqr)|(s>q3+1.5*iqr)).sum())
        rows.append(dict(Variable=col, Type='numeric', N=int(s.count()),
            Missing=int(s.isnull().sum()), Min=float(s.min()), Max=float(s.max()),
            Mean=round(float(s.mean()),3), Outlier_IQR=n_out))
    else:
        rows.append(dict(Variable=col, Type='categorical', N=int(s.count()),
            Missing=int(s.isnull().sum()), Min=np.nan, Max=np.nan,
            Mean=np.nan, Outlier_IQR=np.nan))
pd.DataFrame(rows).to_csv(OUT/"tables/Table_S1_Data_Quality_Audit.csv", index=False)

data.groupby(['District','Year']).size().unstack(fill_value=0)\
    .to_csv(OUT/"tables/Table_S2_District_Year_Counts.csv")
data.groupby('District').size().rename('N_obs')\
    .to_csv(OUT/"tables/Table_S2b_District_Totals.csv")

out_rows=[]
for col in data.select_dtypes(include=np.number).columns:
    if col=='Year': continue
    q1,q3 = data[col].quantile(.25), data[col].quantile(.75); iqr=q3-q1
    lb,ub = q1-1.5*iqr, q3+1.5*iqr
    n_out = int(((data[col]<lb)|(data[col]>ub)).sum())
    out_rows.append(dict(Variable=col, N_outliers=n_out,
        Lower=round(lb,3), Upper=round(ub,3),
        Min=round(float(data[col].min()),3),
        Max=round(float(data[col].max()),3)))
pd.DataFrame(out_rows).to_csv(OUT/"tables/Table_S3_Outlier_Audit.csv", index=False)

In [57]:
# ==============================================================================
# PART 3B — NASA POWER PARAMETER TABLE
# ==============================================================================
pd.DataFrame([
    dict(Variable='Temperature',
         Code='TS (skin temperature)',
         Temporal_aggregation='Annual mean',
         Spatial_assignment='District centroid, 0.5° × 0.625° cell'),
    dict(Variable='Precipitation',
         Code='PRECTOTCORR',
         Temporal_aggregation='Annual mean daily rate (mm day⁻¹)',
         Spatial_assignment='District centroid, 0.5° × 0.625° cell'),
    dict(Variable='Relative humidity',
         Code='RH2M',
         Temporal_aggregation='Annual mean',
         Spatial_assignment='District centroid, 0.5° × 0.625° cell'),
    dict(Variable='Wind speed',
         Code='WS2M',
         Temporal_aggregation='Annual mean',
         Spatial_assignment='District centroid, 0.5° × 0.625° cell'),
    dict(Variable='Solar radiation',
         Code='ALLSKY_SFC_SW_DWN',
         Temporal_aggregation='Annual mean',
         Spatial_assignment='District centroid, 0.5° × 0.625° cell'),
    dict(Variable='Soil wetness',
         Code='GWETROOT',
         Temporal_aggregation='Annual mean',
         Spatial_assignment='District centroid, 0.5° × 0.625° cell'),
]).to_csv(OUT / "tables" / "Table_S8_NASA_POWER_parameters.csv", index=False)
print("\n[NASA POWER TABLE] Table_S8_NASA_POWER_parameters.csv written "
      "(temperature code = TS, grid = 0.5° × 0.625°)")


[NASA POWER TABLE] Table_S8_NASA_POWER_parameters.csv written (temperature code = TS, grid = 0.5° × 0.625°)


In [58]:
# ==============================================================================
# PART 4 — FEATURES / TARGET / GROUPS
# ==============================================================================
FEATURES = [
    'Cultivated_Area_ha','pH','Total_N_percent','Available_P_ppm',
    'Exchangeable_K_ppm','Organic_C_percent','Fertilizer_Used',
    'Soil_Type','Annual_Temperature_C','Precipitation_mm_day',
    'Relative_Humidity_percent','Wind_Speed_m_s',
    'Solar_Radiation_MJ_m2_day','Soil_Wetness'
]
TARGET = 'Production_tons'
X = data[FEATURES].copy()
y = data[TARGET].copy()
years      = data['Year'].values
districts  = data['District'].values
group_keys = pd.Series(districts.astype(str)+"_"+years.astype(str)).values

assert 'District' not in FEATURES and 'Year' not in FEATURES
print(f"Final matrix: X={X.shape}, y={y.shape}")

Final matrix: X=(500, 14), y=(500,)


In [59]:
# ==============================================================================
# PART 5 — CHRONOLOGICAL TRAIN/TEST SPLIT
# ==============================================================================
TRAIN_YEARS = sorted([int(v) for v in np.unique(years) if v <= 2012])
TEST_YEARS  = sorted([int(v) for v in np.unique(years) if v >= 2013])
train_mask  = np.isin(years, TRAIN_YEARS)
test_mask   = np.isin(years, TEST_YEARS)

X_tr = X[train_mask].reset_index(drop=True); y_tr = y[train_mask].reset_index(drop=True)
X_te = X[test_mask].reset_index(drop=True);  y_te = y[test_mask].reset_index(drop=True)
years_tr = years[train_mask];    years_te = years[test_mask]
district_tr = districts[train_mask]; district_te = districts[test_mask]
group_tr = group_keys[train_mask];   group_te = group_keys[test_mask]

print(f"Train: {len(y_tr)} rows, years {min(TRAIN_YEARS)}–{max(TRAIN_YEARS)}")
print(f"Test : {len(y_te)} rows, years {min(TEST_YEARS)}–{max(TEST_YEARS)}")
assert set(TRAIN_YEARS).isdisjoint(set(TEST_YEARS))

Train: 389 rows, years 2000–2012
Test : 111 rows, years 2013–2015


In [60]:
# ==============================================================================
# PART 6 — YEAR-BLOCKED EXPANDING CV (with optional embargo)
# ==============================================================================
def year_blocked_expanding_folds(years_array, n_splits=5, embargo=0):
    """Expanding-window CV on indivisible year blocks; optional year embargo."""
    uniq = np.sort(np.unique(years_array))
    cut  = np.linspace(1, len(uniq)-1, n_splits+1, dtype=int)[1:]
    folds = []
    for b in cut:
        va_yrs = uniq[b:b+1]
        tr_yrs = uniq[:max(b-embargo,1)]
        tr_idx = np.where(np.isin(years_array, tr_yrs))[0]
        va_idx = np.where(np.isin(years_array, va_yrs))[0]
        if len(tr_idx)>0 and len(va_idx)>0:
            folds.append((tr_idx, va_idx))
    return folds

cv_splits = year_blocked_expanding_folds(years_tr, n_splits=5, embargo=0)
fold_rows=[]
for i,(tr,va) in enumerate(cv_splits,1):
    fold_rows.append(dict(
        Fold=i, N_train=len(tr), N_val=len(va),
        Train_Years=f"{years_tr[tr].min()}–{years_tr[tr].max()}",
        Val_Years=f"{years_tr[va].min()}–{years_tr[va].max()}",
        Val_Year_List=sorted(set(years_tr[va].tolist())),
        Years_Disjoint=bool(set(years_tr[tr]).isdisjoint(set(years_tr[va])))
    ))
fold_df = pd.DataFrame(fold_rows)
fold_df.to_csv(OUT/"tables/Table_CV_Fold_Composition.csv", index=False)
print("\n[YEAR-BLOCKED CV FOLDS]")
print(fold_df.to_string(index=False))
assert fold_df['Years_Disjoint'].all()


[YEAR-BLOCKED CV FOLDS]
 Fold  N_train  N_val Train_Years Val_Years Val_Year_List  Years_Disjoint
    1       94     34   2000–2002 2003–2003        [2003]            True
    2      157     38   2000–2004 2005–2005        [2005]            True
    3      222     34   2000–2006 2007–2007        [2007]            True
    4      277     18   2000–2008 2009–2009        [2009]            True
    5      359     30   2000–2011 2012–2012        [2012]            True


In [61]:
# ==============================================================================
# PART 7 — PREPROCESSING
# ==============================================================================
numeric_cols = [c for c in FEATURES if c!='Soil_Type']
categorical_cols = ['Soil_Type']
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore',
                                           sparse_output=False))]), categorical_cols)
])
def mk_pipeline(m): return Pipeline([('pre', preprocessor), ('model', m)])

In [62]:
# ==============================================================================
# PART 8 — MODELS
# ==============================================================================
from xgboost import XGBRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
MODELS = {
    'Ridge': (mk_pipeline(Ridge(random_state=RANDOM_SEED)),
              {'model__alpha':[0.01,0.1,1,10,100]}),
    'ElasticNet': (mk_pipeline(ElasticNet(random_state=RANDOM_SEED, max_iter=20000)),
                   {'model__alpha':[0.001,0.01,0.1,1],
                    'model__l1_ratio':[0.2,0.5,0.8]}),
    'LinearRegression': (mk_pipeline(LinearRegression()), {}),
    'GradientBoosting': (mk_pipeline(GradientBoostingRegressor(random_state=RANDOM_SEED)),
                         {'model__n_estimators':[100,200], 'model__max_depth':[2,3,4]}),
    'XGBoost': (mk_pipeline(XGBRegressor(random_state=RANDOM_SEED, verbosity=0, n_jobs=1)),
                {'model__n_estimators':[100,200], 'model__max_depth':[2,3,4],
                 'model__learning_rate':[0.05,0.1]}),
    'RandomForest': (mk_pipeline(RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)),
                     {'model__n_estimators':[200,400], 'model__max_depth':[None,5,10]}),
    'SVR': (mk_pipeline(SVR(kernel='rbf')),
            {'model__C':[1,10,100], 'model__epsilon':[0.1,0.5,1.0]}),
}


In [63]:
# ==============================================================================
# PART 9 — TRAIN + FOLD SCORES
# ==============================================================================
results = {}
fold_score_rows = []
for name,(est,grid) in MODELS.items():
    print(f"\n[TRAIN] {name}")
    if grid:
        gs = GridSearchCV(est, grid, cv=cv_splits, scoring='r2', n_jobs=-1, refit=True)
        gs.fit(X_tr, y_tr)
        best_est, best_params, n_params = gs.best_estimator_, gs.best_params_, len(grid)
    else:
        best_est = est.fit(X_tr, y_tr); best_params = {}; n_params = 0
    fold_r2 = cross_val_score(clone(best_est), X_tr, y_tr, cv=cv_splits, scoring='r2')
    results[name] = dict(model=best_est, params=best_params, n_params=n_params,
                         fold_r2=fold_r2, cv_mean=float(fold_r2.mean()),
                         cv_se=float(fold_r2.std(ddof=1)/np.sqrt(len(fold_r2))))
    for i,s in enumerate(fold_r2,1):
        fold_score_rows.append(dict(Model=name, Fold=i, R2=round(float(s),5)))

pd.DataFrame(fold_score_rows)\
  .to_csv(OUT/"tables/Table_CV_Fold_Scores_PerModel.csv", index=False)
print("\n[PER-FOLD R²]")
print(pd.DataFrame(fold_score_rows).pivot(index='Model', columns='Fold', values='R2').round(4))


[TRAIN] Ridge

[TRAIN] ElasticNet

[TRAIN] LinearRegression

[TRAIN] GradientBoosting

[TRAIN] XGBoost

[TRAIN] RandomForest

[TRAIN] SVR

[PER-FOLD R²]
Fold                   1       2       3       4       5
Model                                                   
ElasticNet        0.8032  0.8609  0.8978  0.8684  0.8026
GradientBoosting  0.8196  0.8068  0.9268  0.7773  0.8055
LinearRegression  0.7864  0.8612  0.8982  0.8766  0.7901
RandomForest      0.8189  0.8174  0.8957  0.7568  0.7555
Ridge             0.8125  0.8598  0.8980  0.8714  0.7986
SVR               0.2200  0.3585  0.5288  0.3324  0.6014
XGBoost           0.8337  0.8462  0.9269  0.7855  0.8069


In [64]:
# ==============================================================================
# PART 10 — 1-SE + PARSIMONY
# ==============================================================================
best_name = max(results, key=lambda k: results[k]['cv_mean'])
best_mean = results[best_name]['cv_mean']
best_se   = results[best_name]['cv_se']
threshold = best_mean - best_se

sel_rows=[]
for n,r in results.items():
    sel_rows.append(dict(Model=n, CV_R2_Mean=round(r['cv_mean'],5),
        CV_R2_SE=round(r['cv_se'],5), N_tuned_params=r['n_params'],
        Within_1SE=bool(r['cv_mean']>=threshold), Selected=False))
sel_df = pd.DataFrame(sel_rows).sort_values('CV_R2_Mean', ascending=False)
cand = sel_df[sel_df['Within_1SE']].copy()
if cand.empty:
    selected_name = best_name
else:
    cand = cand.sort_values(['N_tuned_params','CV_R2_Mean'], ascending=[True,False])
    selected_name = cand.iloc[0]['Model']
sel_df.loc[sel_df['Model']==selected_name,'Selected'] = True
sel_df.to_csv(OUT/"tables/Table_Model_Selection_1SE.csv", index=False)
print(f"\n[1-SE + PARSIMONY HEURISTIC] best={best_name} "
      f"(CV={best_mean:.4f}±{best_se:.4f}) threshold={threshold:.4f} "
      f"→ selected={selected_name}")
print("  NOTE: 'within 1-SE' is a model-SELECTION HEURISTIC, not a formal test.")
print("  SEs are computed as SD(ddof=1)/sqrt(5) across five temporally")
print("  overlapping expanding-window folds; the folds share training years,")
print("  so SEs are not independent and the 1-SE band is not a confidence")
print("  interval. No claim of statistical indistinguishability is made.")
final_model = results[selected_name]['model']


[1-SE + PARSIMONY HEURISTIC] best=Ridge (CV=0.8481±0.0186) threshold=0.8295 → selected=LinearRegression
  NOTE: 'within 1-SE' is a model-SELECTION HEURISTIC, not a formal test.
  SEs are computed as SD(ddof=1)/sqrt(5) across five temporally
  overlapping expanding-window folds; the folds share training years,
  so SEs are not independent and the 1-SE band is not a confidence
  interval. No claim of statistical indistinguishability is made.


In [65]:
# ==============================================================================
# PART 11 — TEST METRICS
# ==============================================================================
test_rows=[]
for name,r in results.items():
    p = r['model'].predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, p))
    test_rows.append(dict(Model=name,
        Test_R2=round(float(r2_score(y_te,p)),4),
        Test_RMSE=round(float(rmse),2),
        Test_MAE=round(float(mean_absolute_error(y_te,p)),2),
        Test_nRMSE_pct=round(float(rmse/y_te.mean()*100),2),
        Selected=bool(name==selected_name)))
test_df = pd.DataFrame(test_rows).sort_values('Test_R2', ascending=False)
test_df.to_csv(OUT/"tables/Table_Test_Metrics.csv", index=False)
print("\n[TEST METRICS]")
print(test_df.to_string(index=False))
y_pred_te = final_model.predict(X_te)


[TEST METRICS]
           Model  Test_R2  Test_RMSE  Test_MAE  Test_nRMSE_pct  Selected
         XGBoost   0.8559     443.27    340.75           16.58     False
GradientBoosting   0.8527     448.18    341.68           16.77     False
      ElasticNet   0.8419     464.28    357.06           17.37     False
           Ridge   0.8409     465.76    360.17           17.43     False
LinearRegression   0.8378     470.30    365.59           17.60      True
    RandomForest   0.8338     476.12    371.12           17.81     False
             SVR   0.5244     805.32    567.16           30.13     False


In [66]:
# ==============================================================================
# PART 12 — BASELINES + NO-AREA
# ==============================================================================
dummy = DummyRegressor(strategy='mean').fit(X_tr, y_tr)
mean_yield_tr = float((y_tr / X_tr['Cultivated_Area_ha']).mean())
area_ols = LinearRegression().fit(X_tr[['Cultivated_Area_ha']], y_tr)
district_means = pd.Series(y_tr.values).groupby(district_tr).mean()
persistence = pd.Series(district_te).map(district_means)\
                .fillna(float(y_tr.mean())).values
baselines = {
    'Mean-only':               dummy.predict(X_te),
    'Area × train-mean yield': X_te['Cultivated_Area_ha'].values * mean_yield_tr,
    'Area-only OLS':           area_ols.predict(X_te[['Cultivated_Area_ha']]),
    'District persistence':    persistence,
}
base_rows=[dict(Baseline=k,
    R2=round(float(r2_score(y_te,p)),4),
    RMSE=round(float(np.sqrt(mean_squared_error(y_te,p))),2),
    MAE=round(float(mean_absolute_error(y_te,p)),2)) for k,p in baselines.items()]
pd.DataFrame(base_rows).to_csv(OUT/"tables/Table_Baselines.csv", index=False)
print("\n[BASELINES]")
print(pd.DataFrame(base_rows).to_string(index=False))

FEAT_NA = [f for f in FEATURES if f!='Cultivated_Area_ha']
num_na  = [c for c in FEAT_NA if c!='Soil_Type']
pre_na = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num_na),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore',
                                           sparse_output=False))]), ['Soil_Type'])
])
def build_no_area_model():
    p = results[selected_name]['params']
    if selected_name=='Ridge':       return Ridge(alpha=p.get('model__alpha',1.0))
    if selected_name=='ElasticNet':  return ElasticNet(alpha=p.get('model__alpha',0.1),
                                        l1_ratio=p.get('model__l1_ratio',0.5),
                                        random_state=RANDOM_SEED, max_iter=20000)
    if selected_name=='LinearRegression': return LinearRegression()
    if selected_name=='GradientBoosting': return GradientBoostingRegressor(random_state=RANDOM_SEED)
    if selected_name=='RandomForest':     return RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
    if selected_name=='XGBoost':          return XGBRegressor(random_state=RANDOM_SEED, verbosity=0, n_jobs=1)
    return SVR(kernel='rbf')
pipe_na = Pipeline([('pre', pre_na), ('model', build_no_area_model())])
pipe_na.fit(X_tr[FEAT_NA], y_tr)
pred_na = pipe_na.predict(X_te[FEAT_NA])
no_area = dict(Model=f'{selected_name} (no cultivated area)',
    Test_R2=round(float(r2_score(y_te, pred_na)),4),
    Test_RMSE=round(float(np.sqrt(mean_squared_error(y_te, pred_na))),2),
    Test_MAE=round(float(mean_absolute_error(y_te, pred_na)),2))
pd.DataFrame([no_area]).to_csv(OUT/"tables/Table_NoArea_Model.csv", index=False)
print("\n[NO-AREA MODEL]"); print(no_area)


[BASELINES]
               Baseline      R2    RMSE    MAE
              Mean-only -0.0299 1185.04 949.83
Area × train-mean yield  0.7937  530.34 381.59
          Area-only OLS  0.8550  444.63 342.41
   District persistence -0.0191 1178.81 949.26

[NO-AREA MODEL]
{'Model': 'LinearRegression (no cultivated area)', 'Test_R2': 0.6313, 'Test_RMSE': 709.01, 'Test_MAE': 552.67}


In [67]:
# ==============================================================================
# PART 12A — AGGREGATION ROBUSTNESS
# ==============================================================================
print("\n" + "="*80); print("[AGGREGATION ROBUSTNESS]"); print("="*80)
agg_spec = {
    'Production_tons':'sum', 'Cultivated_Area_ha':'sum',
    'pH':'mean','Total_N_percent':'mean','Available_P_ppm':'mean',
    'Exchangeable_K_ppm':'mean','Organic_C_percent':'mean',
    'Annual_Temperature_C':'mean','Precipitation_mm_day':'mean',
    'Relative_Humidity_percent':'mean','Wind_Speed_m_s':'mean',
    'Solar_Radiation_MJ_m2_day':'mean','Soil_Wetness':'mean',
    'Fertilizer_Used':'mean'}
agg = data.groupby(['District','Year']).agg(agg_spec).reset_index()
agg['Soil_Type'] = data.groupby(['District','Year'])['Soil_Type']\
                       .agg(lambda s: s.mode().iloc[0]).values
print(f"Aggregated district-year rows: {len(agg)}")
agg_train = agg[agg['Year'].isin(TRAIN_YEARS)].reset_index(drop=True)
agg_test  = agg[agg['Year'].isin(TEST_YEARS)].reset_index(drop=True)
agg_pipe = clone(final_model)
agg_pipe.fit(agg_train[FEATURES], agg_train[TARGET])
agg_pred = agg_pipe.predict(agg_test[FEATURES])
agg_metrics = dict(Model=f'{selected_name} (district-year aggregated)',
    N_train=len(agg_train), N_test=len(agg_test),
    Test_R2=round(float(r2_score(agg_test[TARGET], agg_pred)),4),
    Test_RMSE=round(float(np.sqrt(mean_squared_error(agg_test[TARGET], agg_pred))),2),
    Test_MAE=round(float(mean_absolute_error(agg_test[TARGET], agg_pred)),2))
pd.DataFrame([agg_metrics]).to_csv(OUT/"tables/Table_S10_Aggregation_Robustness.csv", index=False)
print(agg_metrics)
print("  INTERPRETATION: R² rises because between-district-year variance now "
      "dominates; RMSE/MAE are LARGER in absolute tons because each aggregated "
      "row is a sum over many sub-records. R² and absolute errors are therefore "
      "not directly comparable across row-level and district-year levels.")


[AGGREGATION ROBUSTNESS]
Aggregated district-year rows: 79
{'Model': 'LinearRegression (district-year aggregated)', 'N_train': 64, 'N_test': 15, 'Test_R2': 0.9731, 'Test_RMSE': 1326.22, 'Test_MAE': 1049.25}
  INTERPRETATION: R² rises because between-district-year variance now dominates; RMSE/MAE are LARGER in absolute tons because each aggregated row is a sum over many sub-records. R² and absolute errors are therefore not directly comparable across row-level and district-year levels.


In [68]:
# ==============================================================================
# PART 12B — LODO SENSITIVITY
# ==============================================================================
print("\n" + "="*80); print("[LODO SENSITIVITY]"); print("="*80)
lodo_rows=[]; uniq_districts = sorted(np.unique(districts))
for d in uniq_districts:
    tr_i = np.where(districts!=d)[0]; te_i = np.where(districts==d)[0]
    if len(np.unique(y.values[tr_i]))<2 or len(te_i)<2: continue
    m = clone(final_model).fit(X.iloc[tr_i], y.iloc[tr_i])
    p = m.predict(X.iloc[te_i])
    lodo_rows.append(dict(Held_Out_District=d, N_train=int(len(tr_i)), N_test=int(len(te_i)),
        LODO_R2=round(float(r2_score(y.values[te_i], p)),4),
        LODO_RMSE=round(float(np.sqrt(mean_squared_error(y.values[te_i], p))),2),
        LODO_MAE=round(float(mean_absolute_error(y.values[te_i], p)),2)))
lodo_df = pd.DataFrame(lodo_rows)
lodo_df.to_csv(OUT/"tables/Table_S11_LODO_Sensitivity.csv", index=False)
print(lodo_df.to_string(index=False))
print(f"LODO R² range: {lodo_df['LODO_R2'].min():.4f} – {lodo_df['LODO_R2'].max():.4f}; "
      f"mean={lodo_df['LODO_R2'].mean():.4f} ± {lodo_df['LODO_R2'].std(ddof=1):.4f}")
print("  DESCRIPTIVE only: five folds = five districts sharing within-district "
      "climate/soil, so between-fold variance is expected.")


[LODO SENSITIVITY]
Held_Out_District  N_train  N_test  LODO_R2  LODO_RMSE  LODO_MAE
             Boke      400     100   0.8192     512.25    376.94
        Dara Labu      400     100   0.8342     453.49    341.62
            Habro      400     100   0.8327     497.13    379.64
    Shanan_dhugoo      400     100   0.8507     436.45    328.86
             Tulo      400     100   0.8308     568.24    386.32
LODO R² range: 0.8192 – 0.8507; mean=0.8335 ± 0.0113
  DESCRIPTIVE only: five folds = five districts sharing within-district climate/soil, so between-fold variance is expected.


In [69]:
# ==============================================================================
# PART 13 — ROLLING-ORIGIN PER YEAR
# ==============================================================================
rolling=[]
for yr in TEST_YEARS:
    tr_i = np.where(years<yr)[0]; te_i = np.where(years==yr)[0]
    if len(tr_i)==0 or len(te_i)<3: continue
    m = clone(final_model).fit(X.iloc[tr_i], y.iloc[tr_i])
    p_ = m.predict(X.iloc[te_i])
    rolling.append(dict(Test_Year=int(yr), N_train=len(tr_i), N_test=len(te_i),
        R2=round(float(r2_score(y.iloc[te_i], p_)),4) if len(te_i)>1 else np.nan,
        MAE=round(float(mean_absolute_error(y.iloc[te_i], p_)),2)))
rolling_df = pd.DataFrame(rolling)
rolling_df.to_csv(OUT/"tables/Table_RollingOrigin_PerYear.csv", index=False)
print("\n[ROLLING-ORIGIN PER YEAR]"); print(rolling_df.to_string(index=False))
print("  CAVEAT: with only three test years, per-year R² is unstable; treat "
      "as descriptive.")


[ROLLING-ORIGIN PER YEAR]
 Test_Year  N_train  N_test     R2    MAE
      2013      389      31 0.8293 397.15
      2014      420      32 0.8658 374.98
      2015      452      48 0.8287 338.24
  CAVEAT: with only three test years, per-year R² is unstable; treat as descriptive.


In [70]:
# ==============================================================================
# PART 14 — GROUP-AWARE + YEAR BOOTSTRAP
# ==============================================================================
def group_block_bootstrap(y_true, y_pred, groups, n_boot=2000, seed=RANDOM_SEED):
    rs = np.random.RandomState(seed); uniq = np.unique(groups)
    r2s,rmses,maes = [],[],[]
    for _ in range(n_boot):
        chosen = rs.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([np.where(groups==g)[0] for g in chosen])
        if len(np.unique(y_true[idx]))<2: continue
        r2s.append(r2_score(y_true[idx], y_pred[idx]))
        rmses.append(np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])))
        maes.append(mean_absolute_error(y_true[idx], y_pred[idx]))
    ci = lambda a: (float(np.percentile(a,2.5)), float(np.percentile(a,97.5)))
    return dict(R2_CI=ci(r2s), RMSE_CI=ci(rmses), MAE_CI=ci(maes),
                n_valid=len(r2s), unit='district-year block',
                type='percentile', seed=seed, n_boot=n_boot)
boot = group_block_bootstrap(y_te.values, y_pred_te, group_te)
print("\n[GROUP-AWARE BOOTSTRAP]"); print(json.dumps(boot, indent=2))
pd.DataFrame([
    dict(Metric='R2',   Lower=boot['R2_CI'][0],   Upper=boot['R2_CI'][1],
         Unit=boot['unit'], Type=boot['type'], n_valid=boot['n_valid']),
    dict(Metric='RMSE', Lower=boot['RMSE_CI'][0], Upper=boot['RMSE_CI'][1],
         Unit=boot['unit'], Type=boot['type'], n_valid=boot['n_valid']),
    dict(Metric='MAE',  Lower=boot['MAE_CI'][0],  Upper=boot['MAE_CI'][1],
         Unit=boot['unit'], Type=boot['type'], n_valid=boot['n_valid'])
]).to_csv(OUT/"tables/Table_Bootstrap_GroupAware.csv", index=False)

def year_level_bootstrap(y_true, y_pred, years_arr, n_boot=2000, seed=RANDOM_SEED):
    rs = np.random.RandomState(seed); uniq = np.unique(years_arr); r2s=[]
    for _ in range(n_boot):
        chosen = rs.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([np.where(years_arr==g)[0] for g in chosen])
        if len(np.unique(y_true[idx]))<2: continue
        r2s.append(r2_score(y_true[idx], y_pred[idx]))
    return dict(R2_CI=(float(np.percentile(r2s,2.5)), float(np.percentile(r2s,97.5))),
                n_valid=len(r2s), unit='year block', type='percentile',
                seed=seed, n_boot=n_boot)
boot_yr = year_level_bootstrap(y_te.values, y_pred_te, years_te)
pd.DataFrame([
    dict(Unit=boot['unit'],   R2_Lower=boot['R2_CI'][0], R2_Upper=boot['R2_CI'][1], n_valid=boot['n_valid']),
    dict(Unit=boot_yr['unit'],R2_Lower=boot_yr['R2_CI'][0], R2_Upper=boot_yr['R2_CI'][1], n_valid=boot_yr['n_valid'])
]).to_csv(OUT/"tables/Table_S12_Bootstrap_Sensitivity.csv", index=False)
print("\n[BOOTSTRAP SENSITIVITY — district-year vs year block]")
print(f"  district-year: R2 95% CI = {boot['R2_CI']}")
print(f"  year-level:    R2 95% CI = {boot_yr['R2_CI']}")
print("  CAVEAT: only three test years, so year-level coverage is crude; "
      "reported for transparency, not as a reliable coverage estimate.")


[GROUP-AWARE BOOTSTRAP]
{
  "R2_CI": [
    0.7922368502818647,
    0.869050884745878
  ],
  "RMSE_CI": [
    425.740077562797,
    514.7016063800448
  ],
  "MAE_CI": [
    334.05505502038517,
    394.48381401252067
  ],
  "n_valid": 2000,
  "unit": "district-year block",
  "type": "percentile",
  "seed": 42,
  "n_boot": 2000
}

[BOOTSTRAP SENSITIVITY — district-year vs year block]
  district-year: R2 95% CI = (0.7922368502818647, 0.869050884745878)
  year-level:    R2 95% CI = (0.8242337784364865, 0.8618420036610038)
  CAVEAT: only three test years, so year-level coverage is crude; reported for transparency, not as a reliable coverage estimate.


In [71]:
# ==============================================================================
# PART 15 — CALIBRATION
# ==============================================================================
slope, intercept, r, pval, se = linregress(y_te, y_pred_te)
cal = dict(slope=float(slope), slope_se=float(se),
           intercept=float(intercept), r2=float(r**2))
pd.DataFrame([cal]).to_csv(OUT/"tables/Table_Calibration.csv", index=False)
print(f"\n[CALIBRATION] slope={slope:.4f}±{se:.4f}, intercept={intercept:.2f}, R²={r**2:.4f}")


[CALIBRATION] slope=0.8732±0.0366, intercept=321.85, R²=0.8394


In [72]:
# ==============================================================================
# PART 16 — PERMUTATION IMPORTANCE (association)
# ==============================================================================
perm = permutation_importance(final_model, X_te, y_te, n_repeats=30,
                              random_state=RANDOM_SEED, scoring='r2', n_jobs=-1)
imp = pd.DataFrame({'Feature': FEATURES,
    'Importance_Mean_deltaR2': perm.importances_mean.round(6),
    'Importance_SD': perm.importances_std.round(6)})\
    .sort_values('Importance_Mean_deltaR2', ascending=False)
imp.to_csv(OUT/"tables/Table_Permutation_Importance.csv", index=False)
print("\n[PERMUTATION IMPORTANCE — association, not causation]")
print(imp.to_string(index=False))



[PERMUTATION IMPORTANCE — association, not causation]
                  Feature  Importance_Mean_deltaR2  Importance_SD
       Cultivated_Area_ha                 1.158054       0.117858
                Soil_Type                 0.081672       0.018900
          Fertilizer_Used                 0.009218       0.007284
                       pH                 0.005175       0.006350
        Organic_C_percent                 0.001496       0.006442
          Available_P_ppm                 0.000430       0.000910
Relative_Humidity_percent                 0.000242       0.000247
          Total_N_percent                 0.000144       0.002463
     Precipitation_mm_day                 0.000031       0.000939
     Annual_Temperature_C                 0.000002       0.001363
       Exchangeable_K_ppm                -0.000146       0.005227
Solar_Radiation_MJ_m2_day                -0.000657       0.000977
             Soil_Wetness                -0.001713       0.001566
           Wind_Speed

In [73]:
# ==============================================================================
# PART 17 — LEAKAGE + MANIFEST  [FIX#7, FIX#8, FIX#10, FIX#12, FIX#13]
# ==============================================================================
leak = dict(
    district_in_features=False, year_in_features=False,
    train_test_years_disjoint=bool(set(TRAIN_YEARS).isdisjoint(TEST_YEARS)),
    cv_folds_years_disjoint=bool(fold_df['Years_Disjoint'].all()),
    n_train=int(len(y_tr)), n_test=int(len(y_te)),
    train_years=[min(TRAIN_YEARS), max(TRAIN_YEARS)],
    test_years =[min(TEST_YEARS),  max(TEST_YEARS)],
    lodo_performed=True, lodo_n_districts=int(len(uniq_districts)),
    aggregation_robustness_performed=True,
    bootstrap_sensitivity_performed=True,
    external_validation_performed=False,
    operational_scope='retrospective estimation (same-year area + full-year weather)')
json.dump(leak, open(OUT/"leakage_check.json","w"), indent=2)
print("\n[LEAKAGE CHECK]"); print(json.dumps(leak, indent=2))

data_hash = hashlib.md5(pd.util.hash_pandas_object(data, index=True).values).hexdigest()
manifest = dict(
    random_seed=RANDOM_SEED, data_md5=data_hash, n_rows=int(len(data)),
    features=FEATURES, target=TARGET,
    cv_type='year-blocked expanding window (5 folds, indivisible years)',
    selection_rule='1-SE threshold + parsimony heuristic (not a formal test)',
    selected_model=selected_name,
    selected_params={k:str(v) for k,v in results[selected_name]['params'].items()},
    bootstrap=dict(unit=boot['unit'], type=boot['type'],
                   n_boot=boot['n_boot'], seed=boot['seed']),
    robustness_checks=dict(
        district_year_aggregation=agg_metrics,
        lodo_range=(float(lodo_df['LODO_R2'].min()), float(lodo_df['LODO_R2'].max())),
        bootstrap_sensitivity=dict(district_year=boot['R2_CI'], year=boot_yr['R2_CI'])),
    nasa_power_retrieval_date='2024-11-21',
    # FIX#12: temperature = TS (skin temperature), consistent with manuscript + Table_S8
    nasa_power_parameters=dict(
        temperature='TS', precipitation='PRECTOTCORR',
        relative_humidity='RH2M', wind_speed='WS2M',
        solar_radiation='ALLSKY_SFC_SW_DWN', soil_wetness='GWETROOT'),
    # FIX#13: grid = 0.5° × 0.625° (consistent with manuscript + Table_S8)
    spatial_assignment='district centroid → nearest 0.5°×0.625° NASA POWER grid cell; '
                       'multiple districts may share a cell (documented in Methods)',
    external_validation_performed=False,
    operational_scope='retrospective')
json.dump(manifest, open(OUT/"reproducibility_manifest.json","w"), indent=2)
print("\n[MANIFEST]"); print(json.dumps(manifest, indent=2))



[LEAKAGE CHECK]
{
  "district_in_features": false,
  "year_in_features": false,
  "train_test_years_disjoint": true,
  "cv_folds_years_disjoint": true,
  "n_train": 389,
  "n_test": 111,
  "train_years": [
    2000,
    2012
  ],
  "test_years": [
    2013,
    2015
  ],
  "lodo_performed": true,
  "lodo_n_districts": 5,
  "aggregation_robustness_performed": true,
  "bootstrap_sensitivity_performed": true,
  "external_validation_performed": false,
  "operational_scope": "retrospective estimation (same-year area + full-year weather)"
}

[MANIFEST]
{
  "random_seed": 42,
  "data_md5": "352820e95c947d0da671e4ed66fbe01c",
  "n_rows": 500,
  "features": [
    "Cultivated_Area_ha",
    "pH",
    "Total_N_percent",
    "Available_P_ppm",
    "Exchangeable_K_ppm",
    "Organic_C_percent",
    "Fertilizer_Used",
    "Soil_Type",
    "Annual_Temperature_C",
    "Precipitation_mm_day",
    "Relative_Humidity_percent",
    "Wind_Speed_m_s",
    "Solar_Radiation_MJ_m2_day",
    "Soil_Wetness"
  ],

In [74]:
# ==============================================================================
# PART 18 — FIGURES
# ==============================================================================
# --- Fig. 3: production distribution -----------------------------------------
prod = y.values
fig, ax = plt.subplots(figsize=(7,4.5))
ax.hist(prod, bins='auto', density=True, color='#8DB3D3',
        edgecolor='white', linewidth=0.8, label='Observed data')
if prod.size>=3 and np.ptp(prod)>0:
    kde = gaussian_kde(prod)
    grid = np.linspace(prod.min()*0.95, prod.max()*1.05, 400)
    ax.plot(grid, kde(grid), color='#A33D3D', lw=2, label='Kernel density estimate')
ax.set_xlabel('Annual coffee production (tons)')
ax.set_ylabel('Probability density (tons$^{-1}$)')
ax.spines[['top','right']].set_visible(False); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(OUT/"figures/Fig3_Production_Distribution.png"); plt.close(fig)

In [75]:
# ------------------------------------------------------------------------------
# PART 18B — FIGURE 4: NUMERIC FEATURE DISTRIBUTIONS (raw data, axis units)
# ------------------------------------------------------------------------------
numeric_panels = [
    ('Cultivated_Area_ha',        'Area (ha)'),
    ('pH',                        'pH units'),
    ('Total_N_percent',           'Total N (%)'),
    ('Available_P_ppm',           'Available P (ppm)'),
    ('Exchangeable_K_ppm',        'Exchangeable K (ppm)'),
    ('Organic_C_percent',         'Organic C (%)'),
    ('Annual_Temperature_C',      'Temperature (°C)'),
    ('Precipitation_mm_day',      'Precipitation (mm day$^{-1}$)'),
    ('Relative_Humidity_percent', 'RH (%)'),
    ('Wind_Speed_m_s',            'Wind speed (m s$^{-1}$)'),
    ('Solar_Radiation_MJ_m2_day', 'Solar radiation (MJ m$^{-2}$ day$^{-1}$)'),
    ('Soil_Wetness',              'Soil wetness (dimensionless)'),
    ('Production_tons',           'Production (tons)'),
]
n = len(numeric_panels)
ncols = 4
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows))
axes = axes.flatten()
for ax, (col, label) in zip(axes, numeric_panels):
    ax.boxplot(data[col].dropna(), patch_artist=True, widths=0.5,
               boxprops=dict(facecolor='#8DB3D3', edgecolor='k'),
               medianprops=dict(color='#A33D3D', lw=1.2))
    ax.set_title(col, fontsize=9)
    ax.set_ylabel(label, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
for ax in axes[n:]:
    ax.set_visible(False)
fig.suptitle('Distribution of numeric features (raw data)', fontsize=12, y=1.00)
fig.tight_layout()
fig.savefig(OUT / "figures" / "Fig4_Numeric_Distributions.png",
            dpi=600, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("\n[FIG 4] Numeric distribution grid saved to "
      "figures/Fig4_Numeric_Distributions.png")


[FIG 4] Numeric distribution grid saved to figures/Fig4_Numeric_Distributions.png


In [76]:
# --- Fig. 5: observed vs predicted -------------------------------------------
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(y_te, y_pred_te, s=18, alpha=0.6, edgecolors='k', linewidths=0.3)
lims = [min(y_te.min(),y_pred_te.min())*0.95, max(y_te.max(),y_pred_te.max())*1.05]
ax.plot(lims, lims, 'r--', lw=1, label='1:1 line')
ax.set_xlabel('Observed production (tons)'); ax.set_ylabel('Predicted production (tons)')
ax.set_xlim(lims); ax.set_ylim(lims); ax.legend(frameon=False)
ax.spines[['top','right']].set_visible(False); fig.tight_layout()
fig.savefig(OUT/"figures/Fig5_Observed_vs_Predicted.png"); plt.close(fig)

In [77]:
# --- Fig. 6: permutation importance ------------------------------------------
fig, ax = plt.subplots(figsize=(7,4.8))
ax.barh(imp['Feature'][::-1], imp['Importance_Mean_deltaR2'][::-1],
        xerr=imp['Importance_SD'][::-1], color='#6C8EAD', edgecolor='k')
ax.set_xlabel('Mean decrease in R² (permutation importance)')
ax.set_xscale('symlog', linthresh=1e-3)
ax.set_title('Permutation importance on locked test set (symlog x-axis)')
ax.spines[['top','right']].set_visible(False); fig.tight_layout()
fig.savefig(OUT/"figures/Fig6_Permutation_Importance.png"); plt.close(fig)

In [78]:
# ------------------------------------------------------------------------------
# PART 18D — FIGURE 7: FOUR-PANEL RESIDUAL AND CALIBRATION DIAGNOSTICS
# ------------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from matplotlib.lines import Line2D
resid = y_te.values - y_pred_te
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

# (a) Actual vs. predicted
ax = axes[0, 0]
ax.scatter(y_te, y_pred_te, s=22, alpha=0.65, edgecolors='k', linewidths=0.3)
lims = [min(y_te.min(), y_pred_te.min()) * 0.95,
        max(y_te.max(), y_pred_te.max()) * 1.05]
ax.plot(lims, lims, 'r--', lw=1.2, label='1:1 line')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Observed production (tons)')
ax.set_ylabel('Predicted production (tons)')
ax.set_title('(a) Actual vs. predicted', loc='left')
ax.legend(frameon=False, loc='upper left')
ax.spines[['top', 'right']].set_visible(False)

# (b) Residuals vs. fitted
ax = axes[0, 1]
ax.scatter(y_pred_te, resid, s=22, alpha=0.65, edgecolors='k', linewidths=0.3)
ax.axhline(0, color='r', ls='--', lw=1.2)
ax.set_xlabel('Fitted production (tons)')
ax.set_ylabel('Residual (observed − predicted, tons)')
ax.set_title('(b) Residuals vs. fitted', loc='left')
ax.spines[['top', 'right']].set_visible(False)

# (c) Residual distribution (with normal reference overlay)
ax = axes[1, 0]
ax.hist(resid, bins=20, density=True, color='#8DB3D3',
        edgecolor='white', linewidth=0.8, label='Residuals')
if resid.std(ddof=1) > 0:
    xs = np.linspace(resid.min(), resid.max(), 400)
    ax.plot(xs, norm.pdf(xs, loc=resid.mean(), scale=resid.std(ddof=1)),
            color='#A33D3D', lw=1.5, label='Normal reference')
ax.axvline(0, color='k', ls=':', lw=1)
ax.set_xlabel('Residual (tons)')
ax.set_ylabel('Probability density (tons$^{-1}$)')
ax.set_title('(c) Residual distribution', loc='left')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)

# (d) Year-stratified residuals  [FIX#16: add year-mean legend]
ax = axes[1, 1]
years_te_arr = np.asarray(years_te)
unique_test_years = sorted(np.unique(years_te_arr))
box_data = [resid[years_te_arr == yr] for yr in unique_test_years]
bp = ax.boxplot(box_data, labels=[str(yr) for yr in unique_test_years],
                patch_artist=True, widths=0.5, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='#A33D3D',
                               markeredgecolor='#A33D3D', markersize=5))
for patch in bp['boxes']:
    patch.set_facecolor('#8DB3D3'); patch.set_edgecolor('k')
ax.axhline(0, color='r', ls='--', lw=1.2)
ax.set_xlabel('Test year (Ethiopian Calendar)')
ax.set_ylabel('Residual (tons)')
ax.set_title('(d) Year-stratified residuals', loc='left')
ax.legend(handles=[Line2D([0], [0], marker='D', color='w',
                          markerfacecolor='#A33D3D',
                          markeredgecolor='#A33D3D',
                          markersize=6, label='Year mean')],
          frameon=False, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)

# FIX#17: suptitle echoes selected model + TS/grid
fig.suptitle(
    f'Residual and calibration diagnostics for the selected model '
    f'({selected_name}, locked 2013–2015 EC hold-out)',
    fontsize=12, y=1.02
)
fig.tight_layout()
fig.savefig(OUT / "figures" / "Fig7_Residual_Diagnostics.png",
            dpi=600, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("\n[FIG 7] Four-panel residual diagnostics saved to "
      "figures/Fig7_Residual_Diagnostics.png")
print(f"        Temperature predictor: TS (skin temperature); "
      f"grid 0.5° x 0.625°; see Table_S8_NASA_POWER_parameters.csv")

print("\n" + "="*80)
print("COMPLETE. Outputs in:", OUT.resolve())
print("="*80)


[FIG 7] Four-panel residual diagnostics saved to figures/Fig7_Residual_Diagnostics.png
        Temperature predictor: TS (skin temperature); grid 0.5° x 0.625°; see Table_S8_NASA_POWER_parameters.csv

COMPLETE. Outputs in: /content/publication_outputs_final


In [79]:
# ==============================================================================
# PART 19 — DEPOSIT ASSEMBLY: reconciliation_notes.json + README.md + VERIFICATION
import os, json, hashlib
import pandas as pd

# --- 19A. reconciliation_notes.json -------------------------------------------
# Reconstruct the same evidence object used in Part 1 (kept in memory via
# `unit_note`; rebuild the counts defensively in case the cell was re-run).
_dy_counts = data.groupby(['District', 'Year']).size()
_recon = {
    "file_purpose": "Explains district-identifier reconciliation; supports Section 2.3",
    "data_file": "Coff_Poduction.csv",
    "data_md5": hashlib.md5(
        pd.util.hash_pandas_object(data, index=True).values).hexdigest(),
    "n_rows": int(len(data)),
    "n_districts": int(data['District'].nunique()),
    "year_range": [int(data['Year'].min()), int(data['Year'].max())],
    "recording_unit_note": unit_note,
    "recording_unit_evidence": {
        "unique_district_year_cells": int(len(_dy_counts)),
        "rows_per_district_year": {
            "min": int(_dy_counts.min()),
            "median": int(_dy_counts.median()),
            "max": int(_dy_counts.max()),
        },
        "sub_district_identifier_present_in_csv": False,
    },
    "district_id_available_in_csv": True,
    "district_name_map_applied": {
        "Boke": "Boke",
        "D/Labu": "Dara Labu",
        "Habro": "Habro",
        "Mesela": "Shanan_dhugoo",
        "Tulo": "Tulo",
    },
    "manuscript_section_2_3_claim_no_district_id": False,
    "reconciliation": (
        "Section 2.3 statement that modeling data lack observation-level "
        "district identifiers is INCORRECT: the District column is present and "
        "used for grouping (group_keys, LODO, persistence, Table S2). "
        "Section 3.1 is correct."
    ),
    "aggregation_robustness_table": "tables/Table_S10_Aggregation_Robustness.csv",
}
with open(OUT / "reconciliation_notes.json", "w") as f:
    json.dump(_recon, f, indent=2)
print("[DEPOSIT] reconciliation_notes.json written")

# --- 19B. README.md -----------------------------------------------------------
readme = f"""# Coffee production modelling — reproducibility deposit

## Contents
| # | File | Content | Purpose |
|---|------|---------|---------|
| 1 | `coffee_production_pipeline.ipynb` | Full executable notebook (year-blocked CV folds, 1-SE selection, test metrics, bootstrap, permutation importance, LODO) | Reproducibility |
| 2 | `Coff_Poduction.csv` | Analysis dataset ({len(data)} x {data.shape[1]}); MD5 `{_recon['data_md5']}` | Data version |
| 3 | `reproducibility_manifest.json` | Seed, features, target, CV type, selection rule, selected model, bootstrap details, NASA POWER parameters | Reproducibility |
| 4 | `leakage_check.json` | Confirms no district/year leakage; train/test years disjoint | Reproducibility |
| 5 | `reconciliation_notes.json` | Explains district-identifier reconciliation | Supports Section 2.3 |
| 6 | `README.md` | This guide | Reproducibility |
| 7 | `outputs/` (CSV files) | Regenerated outputs of Tables 3, 4, 6, 7, 8, 9, 10 and S10, S12, S15 | Verification without running code |

## How to run
1. Open `coffee_production_pipeline.ipynb` in Google Colab.
2. Mount Google Drive (`drive.mount('/content/drive')`).
3. Place `Coff_Poduction.csv` in `MyDrive/` (path set in Part 1: `DATA_PATH`).
4. Run all cells. Outputs are written to `{OUT}`.

## Environment
- Python 3, scikit-learn, XGBoost, pandas, numpy, scipy, matplotlib.
- Random seed: `{RANDOM_SEED}` (set in the notebook).
- Selected model: `{selected_name}` (1-SE + parsimony heuristic).

## Notes
- `Coff_Poduction.csv` is the only data input; no external download needed.
- All tables/figures in the manuscript can be regenerated from the notebook.
- Bootstrap CIs: district-year block, percentile, {boot['n_boot']} resamples, seed {boot['seed']}.
"""
with open(OUT / "README.md", "w") as f:
    f.write(readme)
print("[DEPOSIT] README.md written")

# --- 19C. Verification of the full deposit ------------------------------------
print("\n" + "="*80)
print("[DEPOSIT VERIFICATION]")
print("="*80)

REQUIRED_ROOT = [
    "reproducibility_manifest.json",
    "leakage_check.json",
    "reconciliation_notes.json",
    "README.md",
]
for fn in REQUIRED_ROOT:
    p = OUT / fn
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status:8s} {fn}")

# Verify MD5 matches the manifest
manifest = json.load(open(OUT / "reproducibility_manifest.json"))
md5_ok = manifest["data_md5"] == _recon["data_md5"]
print(f"  {'OK' if md5_ok else 'MISMATCH':8s} data MD5 "
      f"(manifest={manifest['data_md5']}, recomputed={_recon['data_md5']})")

# Verify expected output CSVs
EXPECTED_CSVS = {
    "Table_S4_Data_Dictionary.csv":            "tables",
    "Table_S1_Data_Quality_Audit.csv":         "tables",
    "Table_S2_District_Year_Counts.csv":       "tables",
    "Table_S2b_District_Totals.csv":           "tables",
    "Table_S3_Outlier_Audit.csv":              "tables",
    "Table_S8_NASA_POWER_parameters.csv":      "tables",
    "Table_CV_Fold_Composition.csv":           "tables",
    "Table_CV_Fold_Scores_PerModel.csv":       "tables",
    "Table_Model_Selection_1SE.csv":           "tables",
    "Table_Test_Metrics.csv":                  "tables",
    "Table_Baselines.csv":                     "tables",
    "Table_NoArea_Model.csv":                  "tables",
    "Table_S10_Aggregation_Robustness.csv":    "tables",
    "Table_S11_LODO_Sensitivity.csv":          "tables",
    "Table_RollingOrigin_PerYear.csv":         "tables",
    "Table_Bootstrap_GroupAware.csv":          "tables",
    "Table_S12_Bootstrap_Sensitivity.csv":     "tables",
    "Table_Calibration.csv":                   "tables",
    "Table_Permutation_Importance.csv":        "tables",
}
missing = []
for fn, sub in EXPECTED_CSVS.items():
    p = OUT / sub / fn
    if not p.exists():
        missing.append(f"{sub}/{fn}")
print(f"\n  Output CSVs: {len(EXPECTED_CSVS) - len(missing)}/{len(EXPECTED_CSVS)} present")
if missing:
    print("  MISSING CSVs:")
    for m in missing:
        print(f"    - {m}")

# Figures
EXPECTED_FIGS = [
    "Fig3_Production_Distribution.png",
    "Fig4_Numeric_Distributions.png",
    "Fig5_Observed_vs_Predicted.png",
    "Fig6_Permutation_Importance.png",
    "Fig7_Residual_Diagnostics.png",
]
missing_figs = [f for f in EXPECTED_FIGS if not (OUT / "figures" / f).exists()]
print(f"  Figures: {len(EXPECTED_FIGS) - len(missing_figs)}/{len(EXPECTED_FIGS)} present")
if missing_figs:
    print("  MISSING figures:")
    for m in missing_figs:
        print(f"    - figures/{m}")

print("\n[DEPOSIT] Deposit assembly complete. Ready for Zenodo upload.")

[DEPOSIT] reconciliation_notes.json written
[DEPOSIT] README.md written

[DEPOSIT VERIFICATION]
  OK       reproducibility_manifest.json
  OK       leakage_check.json
  OK       reconciliation_notes.json
  OK       README.md
  OK       data MD5 (manifest=352820e95c947d0da671e4ed66fbe01c, recomputed=352820e95c947d0da671e4ed66fbe01c)

  Output CSVs: 19/19 present
  Figures: 5/5 present

[DEPOSIT] Deposit assembly complete. Ready for Zenodo upload.
